# Intercase-Bukhsh pipeline (remaining-time)

LS-ICE-style inter-case ("load state") features added to Bukhsh's remaining-time architecture, scored by real validation-period CC MAE (hyperopt search, same space/budget as the regular Bukhsh HPO). Real-life uses the `ssd` trim, synthetic uses `none`.

## Real

In [ ]:
import sys
import json
import math
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "inter-case-bukhsh"))

from run_experiments_real import _load_data, _resolve_log, BUKHSH_SPACE, BUKHSH_MAX_EVAL
from bukhsh.params import default_params
from trainer import InterCaseBukhshTrainer
from setttings import set_global_seed
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries

set_global_seed(1904)

REAL_DATASETS = [
    "bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
    "bpic20-dom", "bpic20-int", "helpdesk", "sepsis",
]

TRIM_CFG = {"method": "ssd", "frac": 0.60, "k": 1.5, "pct": 0.25}

TOP_N_NEXT = 5

OUT_ROOT = ROOT / "inter_case_bukhsh_output_real"
RESULTS_DIR = ROOT / "results" / "inter_case_bukhsh_real"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODELS = ROOT / "best_models"

print(f"{len(REAL_DATASETS)} real-life datasets: {REAL_DATASETS}")
print(f"trim={TRIM_CFG}  HPO space={BUKHSH_SPACE}  max_evals={BUKHSH_MAX_EVAL}")


def rem_time_to_event_log(rt_df):
    """Turns rem_time.csv predictions into a (caseid, end_timestamp) event log."""
    rt = rt_df.copy()
    rt["start_timestamp"] = pd.to_datetime(rt["start_timestamp"])
    rt["anchor_timestamp"] = pd.to_datetime(rt["anchor_timestamp"])
    rt["predicted_end"] = rt["anchor_timestamp"] + pd.to_timedelta(rt["rem_time_days"], unit="D")
    return pd.concat([
        rt[["caseid", "start_timestamp"]].rename(columns={"start_timestamp": "end_timestamp"}),
        rt[["caseid", "predicted_end"]].rename(columns={"predicted_end": "end_timestamp"}),
    ], ignore_index=True)


def build_kpi_series(event_log, cc_index, tt_index):
    """Matches compile_results.ipynb's own function of the same name exactly."""
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col="end_timestamp", case_col="caseid", window="days", plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col="end_timestamp", case_col="caseid", window="days", plot=False)
    cc_arr = pred_cc.reindex(cc_index).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(tt_index).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr

In [ ]:
import hyperopt


def _score_one_trial(trial_id, trial_cfg, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir):
    """Reloads (or trains) one trial's model, scored by val CC MAE."""
    trial_dir = hpo_dir / trial_id
    empty_val = pd.DataFrame(columns=train_df.columns)
    params = default_params(**trial_cfg)
    trainer = InterCaseBukhshTrainer(
        train_df, empty_val, val_as_test_df, trial_id, params,
        output_dir=trial_dir, use_intercase_features=use_ic, top_n_next=TOP_N_NEXT,
        full_df=full_df,
    )
    t0 = time.perf_counter()
    trainer.run()
    train_s = round(time.perf_counter() - t0, 2)

    rem_df = trainer.predict_rem_time()
    plog = rem_time_to_event_log(rem_df)
    cc_pred, tt_pred = build_kpi_series(plog, cc["val"].index, tt["val"].index)
    cc_mae = mean_absolute_error(cc["val"].to_numpy(), cc_pred)
    tt_mae = mean_absolute_error(tt["val"].to_numpy(), tt_pred)

    row = {**trial_cfg, "trial": trial_id,
          "val_cc_mae": round(cc_mae, 4), "val_tt_mae": round(tt_mae, 4), "train_s": train_s}
    (trial_dir / "result.json").write_text(json.dumps(row))
    return row


def _run_ic_hpo(name, tag, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir):
    """Selects the best hyperparameters for one (dataset, variant) combo, scored by val CC MAE."""
    hpo_dir.mkdir(parents=True, exist_ok=True)
    trials_pkl = hpo_dir / "hyperopt_trials.pkl"
    space = {k: hp.choice(k, v) for k, v in BUKHSH_SPACE.items()}

    hpo_results = []

    if trials_pkl.exists():
        with open(trials_pkl, "rb") as f:
            _loaded = pickle.load(f)
        _ok_docs = [t for t in _loaded.trials if t["result"].get("status") == STATUS_OK]
        _n_stuck = len(_loaded.trials) - len(_ok_docs)
        hpo_trials = Trials()
        if _ok_docs:
            hpo_trials.insert_trial_docs(_ok_docs)
            hpo_trials.refresh()
        print(f"  [{name}/{tag}] resuming: {len(_ok_docs)}/{BUKHSH_MAX_EVAL} trials genuinely completed"
             + (f"  ({_n_stuck} stuck placeholder(s) dropped so fmin() actually continues)" if _n_stuck else ""))
    else:
        print(f"  [{name}/{tag}] HPO starting fresh: {BUKHSH_MAX_EVAL} trials")
        hpo_trials = Trials()

    trial_counter = [len(hpo_trials.trials)]

    def _objective(trial_cfg):
        i = trial_counter[0]
        trial_counter[0] += 1
        trial_id = f"trial_{i:03d}"
        trial_dir = hpo_dir / trial_id
        result_path = trial_dir / "result.json"

        with open(trials_pkl, "wb") as f:
            pickle.dump(hpo_trials, f)

        if result_path.exists():
            cached = json.loads(result_path.read_text())
            print(f"    [{i+1}/{BUKHSH_MAX_EVAL}] cached  val CC MAE={cached['val_cc_mae']:.4f}  ({trial_id})")
            hpo_results.append(cached)
            return {"loss": cached["val_cc_mae"], "status": STATUS_OK}

        print(f"    [{i+1}/{BUKHSH_MAX_EVAL}] {trial_cfg}  ({trial_id})")
        row = _score_one_trial(trial_id, trial_cfg, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir)
        hpo_results.append(row)
        print(f"      val CC MAE={row['val_cc_mae']:.4f}  TT MAE={row['val_tt_mae']:.4f}  time={row['train_s']:.0f}s")
        return {"loss": row["val_cc_mae"], "status": STATUS_OK}

    fmin(fn=_objective, space=space, algo=tpe.suggest,
         max_evals=BUKHSH_MAX_EVAL, trials=hpo_trials, show_progressbar=False)
    with open(trials_pkl, "wb") as f:
        pickle.dump(hpo_trials, f)

    disk_results = []
    for result_path in sorted(hpo_dir.glob("trial_*/result.json")):
        disk_results.append(json.loads(result_path.read_text()))
    if not disk_results:
        raise RuntimeError(f"  [{name}/{tag}] HPO produced zero completed trials -- nothing to select from")
    n_stuck = BUKHSH_MAX_EVAL - len(disk_results)
    if n_stuck > 0:
        print(f"  [{name}/{tag}] {len(disk_results)}/{BUKHSH_MAX_EVAL} trials completed "
             f"({n_stuck} left permanently unresolved by hyperopt's own budget bookkeeping -- harmless, ignored)")

    hpo_df = pd.DataFrame(disk_results).sort_values("val_cc_mae")
    hpo_df.to_csv(hpo_dir.parent / "hpo_results.csv", index=False)

    best_row = hpo_df.iloc[0].to_dict()
    best_params = {k: best_row[k] for k in BUKHSH_SPACE}
    best_params = {
        k: int(v) if k in ("num_heads", "batch_size", "epochs") else float(v)
        for k, v in best_params.items()
    }
    (hpo_dir.parent / "best_params.json").write_text(json.dumps(best_params, indent=2))
    print(f"  [{name}/{tag}] best params: {best_params}  (val CC MAE={best_row['val_cc_mae']:.4f})")
    return best_params

In [ ]:
def score_against_truth(rem_df: pd.DataFrame, true_end: pd.Series) -> dict:
    d = rem_df.copy()
    d["caseid"] = d["caseid"].astype(str)
    d["anchor_timestamp"] = pd.to_datetime(d["anchor_timestamp"])
    d["true_end"] = d["caseid"].map(true_end)
    d = d.dropna(subset=["true_end"])
    true_end_naive = pd.to_datetime(d["true_end"]).dt.tz_localize(None)
    anchor_naive = d["anchor_timestamp"].dt.tz_localize(None)
    d["true_rem_days"] = (true_end_naive - anchor_naive).dt.total_seconds() / 86400
    d["true_rem_days"] = d["true_rem_days"].clip(lower=0.0)
    err = d["rem_time_days"] - d["true_rem_days"]
    return {
        "n_cases": int(len(d)),
        "mae_days": float(np.mean(np.abs(err))),
        "rmse_days": float(np.sqrt(np.mean(err ** 2))),
    }


def _process_dataset(name, tag="intercase"):
    out_dir = OUT_ROOT / name / tag

    log_path = _resolve_log(name)
    cc, tt, train_df, val_df, test_df, val_as_test_df, empty_val = _load_data(log_path, TRIM_CFG)
    full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    true_end = full_df.assign(
        caseid=lambda d: d["caseid"].astype(str)
    ).groupby("caseid")["end_timestamp"].max()

    best_params_path = out_dir / "best_params.json"
    if best_params_path.exists():
        best_params = json.loads(best_params_path.read_text())
        print(f"[{name}/{tag}] best_params.json found -> skipping HPO")
    else:
        best_params = _run_ic_hpo(name, tag, True, train_df, val_as_test_df, full_df, cc, tt, out_dir / "hpo_trials")

    params = default_params(
        epochs=best_params["epochs"], batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"], num_heads=best_params["num_heads"],
    )

    trainer = InterCaseBukhshTrainer(
        train_df, val_df, test_df, f"{name}_test_full_{tag}", params,
        output_dir=out_dir, use_intercase_features=True, top_n_next=TOP_N_NEXT,
        full_df=full_df,
    )

    t0 = time.perf_counter()
    if (out_dir / "meta.pkl").exists():
        print(f"[{name}/{tag}] meta.pkl found -> skipping training")
    else:
        trainer.run()
    train_s = round(time.perf_counter() - t0, 1)

    rem_time_csv = out_dir / "rem_time.csv"
    eval_metrics_json = out_dir / "eval_metrics.json"
    if rem_time_csv.exists() and eval_metrics_json.exists():
        print(f"[{name}/{tag}] rem_time.csv + eval_metrics.json found -> skipping prediction")
        rem_df = pd.read_csv(rem_time_csv)
        window_metrics = json.loads(eval_metrics_json.read_text())
    else:
        rem_df = trainer.predict_rem_time()
        window_metrics = trainer.evaluate()

    truth_metrics = score_against_truth(rem_df, true_end)

    row = {
        "dataset": name,
        "variant": tag,
        "best_params": json.dumps(best_params),
        "train_s": train_s,
        "n_cases_scored": truth_metrics["n_cases"],
        "true_mae_days": truth_metrics["mae_days"],
        "true_rmse_days": truth_metrics["rmse_days"],
        "window_n_prefixes": window_metrics["n_prefixes"],
        "window_mae_days": window_metrics["mae_days"],
        "window_rmse_days": window_metrics["rmse_days"],
    }
    print(f"[{name}/{tag}] true MAE={truth_metrics['mae_days']:.3f}d  "
          f"window MAE={window_metrics['mae_days']:.3f}d  (train {train_s}s, params={best_params})")
    return row


def make_half_prefix_test_df(df, test_df):
    """Truncates each test case to the first ceil(N/2) of its own events."""
    test_cids = set(test_df["caseid"].astype(str))
    df_full = (df[df["caseid"].astype(str).isin(test_cids)]
               .copy().sort_values(["caseid", "end_timestamp"]))
    parts = [grp.iloc[: max(1, math.ceil(len(grp) / 2))] for _, grp in df_full.groupby("caseid", sort=False)]
    return pd.concat(parts, ignore_index=True)


def _setup_regime_dir(out_dir, regime_name):
    """Symlinks the trained model's artifacts into a <regime_name>/ subdir."""
    regime_dir = out_dir / regime_name
    regime_dir.mkdir(exist_ok=True)
    for fname in ("meta.pkl", "train.csv", "vocab_ref.csv"):
        src = (out_dir / fname).resolve()
        dst = regime_dir / fname
        if src.exists() and not dst.exists():
            try:
                dst.symlink_to(src)
            except OSError:
                import shutil
                shutil.copy2(src, dst)
    return regime_dir


def _process_dataset_half(name, tag="intercase"):
    """Half-prefix regime, prediction only."""
    out_dir = OUT_ROOT / name / tag
    if not (out_dir / "meta.pkl").exists():
        print(f"[{name}/{tag}/half] skip -- run _process_dataset(\"{name}\") first")
        return None

    half_dir = _setup_regime_dir(out_dir, "half_prefix")
    rem_time_csv = half_dir / "rem_time.csv"
    if rem_time_csv.exists():
        print(f"[{name}/{tag}/half] rem_time.csv found -> skipping prediction")
        return pd.read_csv(rem_time_csv)

    log_path = _resolve_log(name)
    cc, tt, train_df, val_df, test_df, val_as_test_df, empty_val = _load_data(log_path, TRIM_CFG)
    full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    half_test_df = make_half_prefix_test_df(full_df, test_df)

    best_params = json.loads((out_dir / "best_params.json").read_text())
    params = default_params(
        epochs=best_params["epochs"], batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"], num_heads=best_params["num_heads"],
    )

    trainer = InterCaseBukhshTrainer(
        train_df, empty_val, half_test_df, f"{name}_test_full_{tag}_half", params,
        output_dir=half_dir, use_intercase_features=True, top_n_next=TOP_N_NEXT,
        full_df=full_df,
    )
    rem_df = trainer.predict_rem_time()
    print(f"[{name}/{tag}/half] predicted {len(rem_df)} rows -> {rem_time_csv}")
    return rem_df


def _process_dataset_plain(name, tag="intercase"):
    """Plain-field regime, prediction only."""
    out_dir = OUT_ROOT / name / tag
    if not (out_dir / "meta.pkl").exists():
        print(f"[{name}/{tag}/plain] skip -- run _process_dataset(\"{name}\") first")
        return None

    pf_dir = _setup_regime_dir(out_dir, "pf")
    rem_time_csv = pf_dir / "rem_time.csv"
    if rem_time_csv.exists():
        print(f"[{name}/{tag}/plain] rem_time.csv found -> skipping prediction")
        return pd.read_csv(rem_time_csv)

    sys.path.insert(0, str(ROOT / "plain-field"))
    from arrival import ProphetArrivalModel, compute_arrival_series
    from runner import build_sos_cases, get_inflight_cases
    from sos import (
        empirical_arrival_hour_sampler, most_frequent_first_activity, most_frequent_first_resource,
    )

    log_path = _resolve_log(name)
    cc, tt, train_df, val_df, test_df, val_as_test_df, empty_val = _load_data(log_path, TRIM_CFG)
    full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

    arr_s = compute_arrival_series(full_df)
    vs = pd.Timestamp(cc["val_split"])
    if vs.tzinfo:
        vs = vs.tz_convert(None)
    am = ProphetArrivalModel()
    am.fit(arr_s[arr_s.index < vs])
    pred_arr = am.predict(cc["test"].index)

    fa = most_frequent_first_activity(train_df)
    fr = most_frequent_first_resource(train_df)
    hs = empirical_arrival_hour_sampler(train_df)
    sos_df = build_sos_cases(pred_arr, fa, fr, hs)
    inflight_df = get_inflight_cases(full_df, cc["val_split"])
    print(f"[{name}/{tag}/plain] {len(sos_df)} SOS + {inflight_df['caseid'].nunique()} in-flight cases")

    plain_test_df = pd.concat([sos_df, inflight_df], ignore_index=True)

    _real_events = full_df[["caseid", "task", "end_timestamp"]].copy()
    _real_ts = pd.to_datetime(_real_events["end_timestamp"])
    _real_events["end_timestamp"] = _real_ts.dt.tz_convert(None) if _real_ts.dt.tz is not None else _real_ts
    _sos_events = sos_df[["caseid", "task", "end_timestamp"]].copy()
    _sos_ts = pd.to_datetime(_sos_events["end_timestamp"])
    _sos_events["end_timestamp"] = _sos_ts.dt.tz_convert(None) if _sos_ts.dt.tz is not None else _sos_ts
    full_df_for_plain = pd.concat([_real_events, _sos_events], ignore_index=True)

    best_params = json.loads((out_dir / "best_params.json").read_text())
    params = default_params(
        epochs=best_params["epochs"], batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"], num_heads=best_params["num_heads"],
    )

    trainer = InterCaseBukhshTrainer(
        train_df, empty_val, plain_test_df, f"{name}_test_full_{tag}_plain", params,
        output_dir=pf_dir, use_intercase_features=True, top_n_next=TOP_N_NEXT,
        full_df=full_df_for_plain,
    )
    rem_df = trainer.predict_rem_time()
    print(f"[{name}/{tag}/plain] predicted {len(rem_df)} rows -> {rem_time_csv}")
    return rem_df


def _regime_cc_tt(rt_path, cc_ts, tt_ts):
    """Shared helper: rem_time.csv -> (cc_mae, tt_mae) against the real test series."""
    rt_df = pd.read_csv(rt_path)
    plog = rem_time_to_event_log(rt_df)
    cc_p, tt_p = build_kpi_series(plog, cc_ts["test"].index, tt_ts["test"].index)
    cc_mae = mean_absolute_error(cc_ts["test"].to_numpy(), cc_p)
    tt_mae = mean_absolute_error(tt_ts["test"].to_numpy(), tt_p)
    return round(cc_mae, 3), round(tt_mae, 3)


def _dataset_summary_row(name, tag="intercase"):
    """One row: best_params + rem_time MAE + CC/TT MAE for all three regimes."""
    out_dir = OUT_ROOT / name / tag
    best_params_path = out_dir / "best_params.json"
    rem_time_csv = out_dir / "rem_time.csv"
    eval_metrics_json = out_dir / "eval_metrics.json"

    row = {"dataset": name, "best_params": None,
           "true_mae_days": None, "window_mae_days": None}
    for regime in ["", "_half", "_plain"]:
        for series in ["cc", "tt"]:
            row[f"{series}_mae__baseline{regime}"] = None
            row[f"{series}_mae__intercase{regime}"] = None
            row[f"{series}_mae_pct_change{regime}"] = None

    if not (best_params_path.exists() and rem_time_csv.exists() and eval_metrics_json.exists()):
        return row

    log_path = _resolve_log(name)
    cc_ts, tt_ts, train_df, val_df, test_df, val_as_test_df, empty_val = _load_data(log_path, TRIM_CFG)
    full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    true_end = full_df.assign(caseid=lambda d: d["caseid"].astype(str)).groupby("caseid")["end_timestamp"].max()

    row["best_params"] = best_params_path.read_text().strip()
    intercase_rt = pd.read_csv(rem_time_csv)
    row["true_mae_days"] = round(score_against_truth(intercase_rt, true_end)["mae_days"], 3)
    row["window_mae_days"] = round(json.loads(eval_metrics_json.read_text())["mae_days"], 3)

    base_dir = BEST_MODELS / name / "ssd" / "bukhsh" / f"{name}_test_full"
    regime_paths = {
        "": (base_dir / "rem_time.csv", rem_time_csv),
        "_half": (base_dir / "half_prefix" / "rem_time.csv", out_dir / "half_prefix" / "rem_time.csv"),
        "_plain": (base_dir / "pf" / "rem_time.csv", out_dir / "pf" / "rem_time.csv"),
    }
    for regime, (base_path, ic_path) in regime_paths.items():
        if not (base_path.exists() and ic_path.exists()):
            continue
        cc_b, tt_b = _regime_cc_tt(base_path, cc_ts, tt_ts)
        cc_i, tt_i = _regime_cc_tt(ic_path, cc_ts, tt_ts)
        row[f"cc_mae__baseline{regime}"] = cc_b
        row[f"cc_mae__intercase{regime}"] = cc_i
        row[f"cc_mae_pct_change{regime}"] = round((cc_i - cc_b) / cc_b * 100, 2)
        row[f"tt_mae__baseline{regime}"] = tt_b
        row[f"tt_mae__intercase{regime}"] = tt_i
        row[f"tt_mae_pct_change{regime}"] = round((tt_i - tt_b) / tt_b * 100, 2)

    return row


def print_summary_table():
    df = pd.DataFrame([_dataset_summary_row(ds) for ds in REAL_DATASETS])
    df.to_csv(RESULTS_DIR / "summary.csv", index=False)
    n_done = df["true_mae_days"].notna().sum()
    n_half = df["cc_mae__intercase_half"].notna().sum()
    n_plain = df["cc_mae__intercase_plain"].notna().sum()
    print(f"\n--- summary: {n_done}/{len(df)} full complete, {n_half}/{len(df)} half complete, "
          f"{n_plain}/{len(df)} plain complete ---")
    print(df.to_string(index=False))
    return df

### 1. First (full-trace)

In [ ]:
rows = []
for name in tqdm(REAL_DATASETS, desc="dataset (intercase)"):
    rows.append(_process_dataset(name))
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "raw_results.csv", index=False)
results_df = pd.DataFrame(rows)
results_df


### 2. Half-prefix

In [ ]:
for name in tqdm(REAL_DATASETS, desc="dataset (intercase, half)"):
    _process_dataset_half(name)


### 3. Plain-field

In [ ]:
for name in tqdm(REAL_DATASETS, desc="dataset (intercase, plain)"):
    _process_dataset_plain(name)


### Summary

In [ ]:
print_summary_table()


## Synthetic

In [ ]:
import sys
import json
import math
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import mean_absolute_error

ROOT = Path.cwd().resolve().parent.parent  # pipelines/intercase/ -> repo root
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "inter-case-bukhsh"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

from ts_comparison import load_splits
from create_prefixes_from_windows import make_three_way_split
from bukhsh.params import default_params
from trainer import InterCaseBukhshTrainer
from setttings import set_global_seed
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries

set_global_seed(1904)

EXCLUDE_DATASETS = {"loan_recency", "o2c_recency"}
SYNTH_DATASETS = sorted(
    f.stem for f in (ROOT / "data" / "synthetic").glob("*.xes")
    if f.stem not in EXCLUDE_DATASETS
)

BUKHSH_SPACE = {
    "num_heads":     [2, 4],
    "batch_size":    [16, 32],
    "epochs":        [20, 50],
    "learning_rate": [0.0005, 0.001, 0.005],
}
BUKHSH_MAX_EVAL = 12
TOP_N_NEXT = 5

OUT_ROOT = ROOT / "inter_case_bukhsh_output_synthetic"
RESULTS_DIR = ROOT / "results" / "inter_case_bukhsh_synthetic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODELS = ROOT / "best_models"

print(f"{len(SYNTH_DATASETS)} synthetic datasets: {SYNTH_DATASETS}")
print(f"HPO space={BUKHSH_SPACE}  max_evals={BUKHSH_MAX_EVAL}")


In [ ]:
import hyperopt


def _score_one_trial(trial_id, trial_cfg, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir):
    """Reloads (or trains) one trial's model, scored by val CC MAE."""
    trial_dir = hpo_dir / trial_id
    empty_val = pd.DataFrame(columns=train_df.columns)
    params = default_params(**trial_cfg)
    trainer = InterCaseBukhshTrainer(
        train_df, empty_val, val_as_test_df, trial_id, params,
        output_dir=trial_dir, use_intercase_features=use_ic, top_n_next=TOP_N_NEXT,
        full_df=full_df,
    )
    t0 = time.perf_counter()
    trainer.run()
    train_s = round(time.perf_counter() - t0, 2)

    rem_df = trainer.predict_rem_time()
    plog = rem_time_to_event_log(rem_df)
    cc_pred, tt_pred = build_kpi_series(plog, cc["val"].index, tt["val"].index)
    cc_mae = mean_absolute_error(cc["val"].to_numpy(), cc_pred)
    tt_mae = mean_absolute_error(tt["val"].to_numpy(), tt_pred)

    row = {**trial_cfg, "trial": trial_id,
          "val_cc_mae": round(cc_mae, 4), "val_tt_mae": round(tt_mae, 4), "train_s": train_s}
    (trial_dir / "result.json").write_text(json.dumps(row))
    return row


def _run_ic_hpo(name, tag, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir):
    """Selects the best hyperparameters for one (dataset, variant) combo, scored by val CC MAE."""
    hpo_dir.mkdir(parents=True, exist_ok=True)
    trials_pkl = hpo_dir / "hyperopt_trials.pkl"
    space = {k: hp.choice(k, v) for k, v in BUKHSH_SPACE.items()}

    hpo_results = []

    if trials_pkl.exists():
        with open(trials_pkl, "rb") as f:
            hpo_trials = pickle.load(f)
        print(f"  [{name}/{tag}] found {len(hpo_trials.trials)} existing trials -> rescoring (no retraining)")

        for i, t in enumerate(hpo_trials.trials):
            trial_id = f"trial_{i:03d}"
            trial_dir = hpo_dir / trial_id
            result_path = trial_dir / "result.json"

            if result_path.exists():
                cached = json.loads(result_path.read_text())
                print(f"    [{trial_id}] cached  val CC MAE={cached['val_cc_mae']:.4f}")
                hpo_results.append(cached)
                continue

            if not (trial_dir / "meta.pkl").exists():
                print(f"    [{trial_id}] skip -- no trained weights on disk")
                continue

            vals = {k: v[0] for k, v in t["misc"]["vals"].items() if v}
            if not vals:
                print(f"    [{trial_id}] skip -- incomplete trial record ({t['result']})")
                continue
            trial_cfg = hyperopt.space_eval(space, vals)

            row = _score_one_trial(trial_id, trial_cfg, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir)
            hpo_results.append(row)
            print(f"    [{trial_id}] rescored  val CC MAE={row['val_cc_mae']:.4f}  cfg={trial_cfg}")

    else:
        print(f"  [{name}/{tag}] HPO starting: {BUKHSH_MAX_EVAL} trials")
        hpo_trials = Trials()
        trial_counter = [0]

        def _objective(trial_cfg):
            i = trial_counter[0]
            trial_counter[0] += 1
            trial_id = f"trial_{i:03d}"

            with open(trials_pkl, "wb") as f:
                pickle.dump(hpo_trials, f)

            print(f"    [{i+1}/{BUKHSH_MAX_EVAL}] {trial_cfg}  ({trial_id})")
            row = _score_one_trial(trial_id, trial_cfg, use_ic, train_df, val_as_test_df, full_df, cc, tt, hpo_dir)
            hpo_results.append(row)
            print(f"      val CC MAE={row['val_cc_mae']:.4f}  TT MAE={row['val_tt_mae']:.4f}  time={row['train_s']:.0f}s")
            return {"loss": row["val_cc_mae"], "status": STATUS_OK}

        fmin(fn=_objective, space=space, algo=tpe.suggest,
             max_evals=BUKHSH_MAX_EVAL, trials=hpo_trials, show_progressbar=False)
        with open(trials_pkl, "wb") as f:
            pickle.dump(hpo_trials, f)

    hpo_df = pd.DataFrame(hpo_results).sort_values("val_cc_mae")
    hpo_df.to_csv(hpo_dir.parent / "hpo_results.csv", index=False)

    best_row = hpo_df.iloc[0].to_dict()
    best_params = {k: best_row[k] for k in BUKHSH_SPACE}
    best_params = {
        k: int(v) if k in ("num_heads", "batch_size", "epochs") else float(v)
        for k, v in best_params.items()
    }
    (hpo_dir.parent / "best_params.json").write_text(json.dumps(best_params, indent=2))
    print(f"  [{name}/{tag}] best params: {best_params}  (val CC MAE={best_row['val_cc_mae']:.4f})")
    return best_params

In [ ]:
def score_against_truth(rem_df: pd.DataFrame, true_end: pd.Series) -> dict:
    d = rem_df.copy()
    d["caseid"] = d["caseid"].astype(str)
    d["anchor_timestamp"] = pd.to_datetime(d["anchor_timestamp"])
    d["true_end"] = d["caseid"].map(true_end)
    d = d.dropna(subset=["true_end"])
    true_end_naive = pd.to_datetime(d["true_end"]).dt.tz_localize(None)
    anchor_naive = d["anchor_timestamp"].dt.tz_localize(None)
    d["true_rem_days"] = (true_end_naive - anchor_naive).dt.total_seconds() / 86400
    d["true_rem_days"] = d["true_rem_days"].clip(lower=0.0)
    err = d["rem_time_days"] - d["true_rem_days"]
    return {
        "n_cases": int(len(d)),
        "mae_days": float(np.mean(np.abs(err))),
        "rmse_days": float(np.sqrt(np.mean(err ** 2))),
    }


def _process_dataset(name, tag="intercase"):
    out_dir = OUT_ROOT / name / tag

    split = load_splits(name, "none", is_real=False)
    train_df, val_df, test_df, full_df = split["train"], split["val"], split["test"], split["df"]
    true_end = full_df.assign(
        caseid=lambda d: d["caseid"].astype(str)
    ).groupby("caseid")["end_timestamp"].max()

    _, _, val_as_test_df = make_three_way_split(
        full_df, case_col="caseid", time_col="end_timestamp",
        train_split=split["cc"]["train_split"], val_split=split["cc"]["train_split"],
        full_traces=True,
    )

    best_params_path = out_dir / "best_params.json"
    if best_params_path.exists():
        best_params = json.loads(best_params_path.read_text())
        print(f"[{name}/{tag}] best_params.json found -> skipping HPO")
    else:
        best_params = _run_ic_hpo(name, tag, True, train_df, val_as_test_df, full_df, split["cc"], split["tt"], out_dir / "hpo_trials")

    params = default_params(
        epochs=best_params["epochs"], batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"], num_heads=best_params["num_heads"],
    )

    trainer = InterCaseBukhshTrainer(
        train_df, val_df, test_df, f"{name}_test_full_{tag}", params,
        output_dir=out_dir, use_intercase_features=True, top_n_next=TOP_N_NEXT,
        full_df=full_df,
    )

    t0 = time.perf_counter()
    if (out_dir / "meta.pkl").exists():
        print(f"[{name}/{tag}] meta.pkl found -> skipping training")
    else:
        trainer.run()
    train_s = round(time.perf_counter() - t0, 1)

    rem_time_csv = out_dir / "rem_time.csv"
    eval_metrics_json = out_dir / "eval_metrics.json"
    if rem_time_csv.exists() and eval_metrics_json.exists():
        print(f"[{name}/{tag}] rem_time.csv + eval_metrics.json found -> skipping prediction")
        rem_df = pd.read_csv(rem_time_csv)
        window_metrics = json.loads(eval_metrics_json.read_text())
    else:
        rem_df = trainer.predict_rem_time()
        window_metrics = trainer.evaluate()

    truth_metrics = score_against_truth(rem_df, true_end)

    row = {
        "dataset": name,
        "variant": tag,
        "best_params": json.dumps(best_params),
        "train_s": train_s,
        "n_cases_scored": truth_metrics["n_cases"],
        "true_mae_days": truth_metrics["mae_days"],
        "true_rmse_days": truth_metrics["rmse_days"],
        "window_n_prefixes": window_metrics["n_prefixes"],
        "window_mae_days": window_metrics["mae_days"],
        "window_rmse_days": window_metrics["rmse_days"],
    }
    print(f"[{name}/{tag}] true MAE={truth_metrics['mae_days']:.3f}d  "
          f"window MAE={window_metrics['mae_days']:.3f}d  (train {train_s}s, params={best_params})")
    return row


def make_half_prefix_test_df(df, test_df):
    """Truncates each test case to the first ceil(N/2) of its own events."""
    test_cids = set(test_df["caseid"].astype(str))
    df_full = (df[df["caseid"].astype(str).isin(test_cids)]
               .copy().sort_values(["caseid", "end_timestamp"]))
    parts = [grp.iloc[: max(1, math.ceil(len(grp) / 2))] for _, grp in df_full.groupby("caseid", sort=False)]
    return pd.concat(parts, ignore_index=True)


def _setup_regime_dir(out_dir, regime_name):
    """Symlinks the trained model's artifacts into a <regime_name>/ subdir."""
    regime_dir = out_dir / regime_name
    regime_dir.mkdir(exist_ok=True)
    for fname in ("meta.pkl", "train.csv", "vocab_ref.csv"):
        src = (out_dir / fname).resolve()
        dst = regime_dir / fname
        if src.exists() and not dst.exists():
            try:
                dst.symlink_to(src)
            except OSError:
                import shutil
                shutil.copy2(src, dst)
    return regime_dir


def _process_dataset_half(name, tag="intercase"):
    """Half-prefix regime, prediction only."""
    out_dir = OUT_ROOT / name / tag
    if not (out_dir / "meta.pkl").exists():
        print(f"[{name}/{tag}/half] skip -- run _process_dataset(\"{name}\") first")
        return None

    half_dir = _setup_regime_dir(out_dir, "half_prefix")
    rem_time_csv = half_dir / "rem_time.csv"
    if rem_time_csv.exists():
        print(f"[{name}/{tag}/half] rem_time.csv found -> skipping prediction")
        return pd.read_csv(rem_time_csv)

    split = load_splits(name, "none", is_real=False)
    train_df, test_df, full_df = split["train"], split["test"], split["df"]
    half_test_df = make_half_prefix_test_df(full_df, test_df)

    best_params = json.loads((out_dir / "best_params.json").read_text())
    params = default_params(
        epochs=best_params["epochs"], batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"], num_heads=best_params["num_heads"],
    )
    empty_val = pd.DataFrame(columns=train_df.columns)

    trainer = InterCaseBukhshTrainer(
        train_df, empty_val, half_test_df, f"{name}_test_full_{tag}_half", params,
        output_dir=half_dir, use_intercase_features=True, top_n_next=TOP_N_NEXT,
        full_df=full_df,
    )
    rem_df = trainer.predict_rem_time()
    print(f"[{name}/{tag}/half] predicted {len(rem_df)} rows -> {rem_time_csv}")
    return rem_df


def _process_dataset_plain(name, tag="intercase"):
    """Plain-field regime, prediction only."""
    out_dir = OUT_ROOT / name / tag
    if not (out_dir / "meta.pkl").exists():
        print(f"[{name}/{tag}/plain] skip -- run _process_dataset(\"{name}\") first")
        return None

    pf_dir = _setup_regime_dir(out_dir, "pf")
    rem_time_csv = pf_dir / "rem_time.csv"
    if rem_time_csv.exists():
        print(f"[{name}/{tag}/plain] rem_time.csv found -> skipping prediction")
        return pd.read_csv(rem_time_csv)

    from arrival import ProphetArrivalModel, compute_arrival_series
    from runner import build_sos_cases, get_inflight_cases
    from sos import (
        empirical_arrival_hour_sampler, most_frequent_first_activity, most_frequent_first_resource,
    )

    split = load_splits(name, "none", is_real=False)
    train_df, full_df, cc = split["train"], split["df"], split["cc"]

    arr_s = compute_arrival_series(full_df)
    vs = pd.Timestamp(cc["val_split"])
    if vs.tzinfo:
        vs = vs.tz_convert(None)
    am = ProphetArrivalModel()
    am.fit(arr_s[arr_s.index < vs])
    pred_arr = am.predict(cc["test"].index)

    fa = most_frequent_first_activity(train_df)
    fr = most_frequent_first_resource(train_df)
    hs = empirical_arrival_hour_sampler(train_df)
    sos_df = build_sos_cases(pred_arr, fa, fr, hs)
    inflight_df = get_inflight_cases(full_df, cc["val_split"])
    print(f"[{name}/{tag}/plain] {len(sos_df)} SOS + {inflight_df['caseid'].nunique()} in-flight cases")

    plain_test_df = pd.concat([sos_df, inflight_df], ignore_index=True)

    _real_events = full_df[["caseid", "task", "end_timestamp"]].copy()
    _real_ts = pd.to_datetime(_real_events["end_timestamp"])
    _real_events["end_timestamp"] = _real_ts.dt.tz_convert(None) if _real_ts.dt.tz is not None else _real_ts
    _sos_events = sos_df[["caseid", "task", "end_timestamp"]].copy()
    _sos_ts = pd.to_datetime(_sos_events["end_timestamp"])
    _sos_events["end_timestamp"] = _sos_ts.dt.tz_convert(None) if _sos_ts.dt.tz is not None else _sos_ts
    full_df_for_plain = pd.concat([_real_events, _sos_events], ignore_index=True)

    best_params = json.loads((out_dir / "best_params.json").read_text())
    params = default_params(
        epochs=best_params["epochs"], batch_size=best_params["batch_size"],
        learning_rate=best_params["learning_rate"], num_heads=best_params["num_heads"],
    )
    empty_val = pd.DataFrame(columns=train_df.columns)

    trainer = InterCaseBukhshTrainer(
        train_df, empty_val, plain_test_df, f"{name}_test_full_{tag}_plain", params,
        output_dir=pf_dir, use_intercase_features=True, top_n_next=TOP_N_NEXT,
        full_df=full_df_for_plain,
    )
    rem_df = trainer.predict_rem_time()
    print(f"[{name}/{tag}/plain] predicted {len(rem_df)} rows -> {rem_time_csv}")
    return rem_df


def _regime_cc_tt(rt_path, cc_ts, tt_ts):
    """rem_time.csv -> (cc_mae, tt_mae) against the real test series."""
    rt_df = pd.read_csv(rt_path)
    plog = rem_time_to_event_log(rt_df)
    cc_p, tt_p = build_kpi_series(plog, cc_ts["test"].index, tt_ts["test"].index)
    cc_mae = mean_absolute_error(cc_ts["test"].to_numpy(), cc_p)
    tt_mae = mean_absolute_error(tt_ts["test"].to_numpy(), tt_p)
    return round(cc_mae, 3), round(tt_mae, 3)


def _dataset_summary_row(name, tag="intercase"):
    """One row: best_params + rem_time MAE + CC/TT MAE for all three regimes."""
    out_dir = OUT_ROOT / name / tag
    best_params_path = out_dir / "best_params.json"
    rem_time_csv = out_dir / "rem_time.csv"
    eval_metrics_json = out_dir / "eval_metrics.json"

    row = {"dataset": name, "best_params": None,
           "true_mae_days": None, "window_mae_days": None}
    for regime in ["", "_half", "_plain"]:
        for series in ["cc", "tt"]:
            row[f"{series}_mae__baseline{regime}"] = None
            row[f"{series}_mae__intercase{regime}"] = None
            row[f"{series}_mae_pct_change{regime}"] = None

    if not (best_params_path.exists() and rem_time_csv.exists() and eval_metrics_json.exists()):
        return row

    split = load_splits(name, "none", is_real=False)
    full_df = split["df"]
    true_end = full_df.assign(caseid=lambda d: d["caseid"].astype(str)).groupby("caseid")["end_timestamp"].max()

    row["best_params"] = best_params_path.read_text().strip()
    intercase_rt = pd.read_csv(rem_time_csv)
    row["true_mae_days"] = round(score_against_truth(intercase_rt, true_end)["mae_days"], 3)
    row["window_mae_days"] = round(json.loads(eval_metrics_json.read_text())["mae_days"], 3)

    cc_ts, tt_ts = split["cc"], split["tt"]
    base_dir = BEST_MODELS / name / "bukhsh" / f"{name}_test_full"
    regime_paths = {
        "": (base_dir / "rem_time.csv", rem_time_csv),
        "_half": (base_dir / "half_prefix" / "rem_time.csv", out_dir / "half_prefix" / "rem_time.csv"),
        "_plain": (base_dir / "pf" / "rem_time.csv", out_dir / "pf" / "rem_time.csv"),
    }
    for regime, (base_path, ic_path) in regime_paths.items():
        if not (base_path.exists() and ic_path.exists()):
            continue
        cc_b, tt_b = _regime_cc_tt(base_path, cc_ts, tt_ts)
        cc_i, tt_i = _regime_cc_tt(ic_path, cc_ts, tt_ts)
        row[f"cc_mae__baseline{regime}"] = cc_b
        row[f"cc_mae__intercase{regime}"] = cc_i
        row[f"cc_mae_pct_change{regime}"] = round((cc_i - cc_b) / cc_b * 100, 2)
        row[f"tt_mae__baseline{regime}"] = tt_b
        row[f"tt_mae__intercase{regime}"] = tt_i
        row[f"tt_mae_pct_change{regime}"] = round((tt_i - tt_b) / tt_b * 100, 2)

    return row


def print_summary_table():
    df = pd.DataFrame([_dataset_summary_row(ds) for ds in SYNTH_DATASETS])
    df.to_csv(RESULTS_DIR / "summary.csv", index=False)
    n_done = df["true_mae_days"].notna().sum()
    n_half = df["cc_mae__intercase_half"].notna().sum()
    n_plain = df["cc_mae__intercase_plain"].notna().sum()
    print(f"\n--- summary: {n_done}/{len(df)} full complete, {n_half}/{len(df)} half complete, "
          f"{n_plain}/{len(df)} plain complete ---")
    print(df.to_string(index=False))
    return df

### 1. First (full-trace)

In [ ]:
rows = []
for name in tqdm(SYNTH_DATASETS, desc="dataset (intercase-bukhsh)"):
    rows.append(_process_dataset(name))
    pd.DataFrame(rows).to_csv(RESULTS_DIR / "raw_results.csv", index=False)
results_df = pd.DataFrame(rows)
results_df


### 2. Half-prefix

In [ ]:
for name in tqdm(SYNTH_DATASETS, desc="dataset (intercase-bukhsh, half)"):
    _process_dataset_half(name)


### 3. Plain-field

In [ ]:
for name in tqdm(SYNTH_DATASETS, desc="dataset (intercase-bukhsh, plain)"):
    _process_dataset_plain(name)


### Summary

In [ ]:
print_summary_table()
